# narrative_dna Colab Quickstart

Este notebook muestra el flujo JSON-first de `narrative_dna`: segmentación, anotación heurística, clasificación LLM estructurada, adjudicación conservadora, auditoría, relaciones/cadenas y tiempos por etapa.

La idea central: el JSON es la fuente de verdad; la notación compacta se deriva del JSON validado.


## 1. Clonar el repo desde GitHub

Para revisar la versión de este PR, la rama por defecto es `codex/add-llm-timing-logs`. Si ya está mergeada y quieres usar `main`, cambia `REPO_BRANCH` a `main`.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/jcval94/ADNarrativa.git"
REPO_BRANCH = os.environ.get("ADNARRATIVA_BRANCH", "codex/add-llm-timing-logs")
REPO_DIR = Path("/content/ADNarrativa")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

os.chdir(REPO_DIR)
branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("repo:", Path.cwd())
print("branch:", branch)
print("commit:", commit)

## 2. Instalar el paquete


In [ ]:
%pip install -q -e ".[dev]"

## 3. Importar módulos principales


In [ ]:
import json
import sys
from pathlib import Path

REPO_DIR = Path("/content/ADNarrativa")
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from narrative_dna.loader import load_text_document  # noqa: E402
from narrative_dna.pipeline import run_pipeline, run_pipeline_from_text  # noqa: E402

print("Imports OK")

## 4. Caso de prueba exigente

Este ejemplo no es una frase fácil. Mezcla analogía, tesis, ejemplo simulado, riesgo, regla de gobierno, instrucciones, utilidad y preguntas al lector. Es mejor para ver si el repo realmente captura significado narrativo y no sólo signos de interrogación.


In [ ]:
transcript_text = "\n".join(
    [
        "Imagina un hospital peque?o que quiere usar IA para priorizar llamadas de pacientes.",
        "El director cree que el sistema debe ser una br?jula, no un piloto autom?tico.",
        "En una prueba simulada, diez expedientes con s?ntomas urgentes suben al inicio",
        "de la cola y tres casos ambiguos quedan marcados para revisi?n humana.",
        "Pero hay un riesgo serio: si los datos hist?ricos tienen sesgos,",
        "la IA puede repetirlos con apariencia de objetividad.",
        "Por eso la regla de gobierno debe ser simple: la IA recomienda,",
        "el equipo cl?nico decide y cada cambio queda registrado.",
        "Primero define qu? decisi?n se automatiza; despu?s mide falsos positivos",
        "y falsos negativos; finalmente publica un reporte mensual de errores.",
        "?A qui?n ayuda este sistema? ?A qui?n podr?a dejar fuera?",
        "?Qu? evidencia necesitar?as antes de confiar en ?l?",
    ]
)

# Texto corto que motiv? la prueba original. ?salo para comparar si quieres.
black_box_memory_text = "\n".join(
    [
        "Los aviones tienen una caja negra. Es un registro m?nimo que guarda lo esencial.",
        "Quiz?s podemos hacer algo parecido con nuestra memoria.",
        "Te propongo algo simple: una vez al a?o, crea tu propia caja negra emocional.",
        "?Qu? aprendiste este a?o? ?Qu? lograste?",
        "?Qu? quieres dejar registrado para siempre?",
    ]
)

document = load_text_document(
    transcript_text,
    document_id="colab_challenging_demo",
    source_path="<colab-string>",
    metadata={"source": "colab_challenging_example"},
    language="es",
)

print("document_id:", document.document_id)
print("units:", len(document.units))
for unit in document.units:
    print(unit.unit_id, unit.final_notation, "->", unit.text)

## 5. Ejecutar el pipeline JSON-first sin LLM

Este modo conserva la precisión: no inventa etiquetas finales. Genera unidades y candidatos heurísticos auditables, pero deja la notación final en `N_N0{0}` salvo reglas determinísticas.


In [ ]:
result = run_pipeline_from_text(
    transcript_text,
    document_id="colab_challenging_demo",
    source_path="<colab-string>",
    output_dir="outputs",
    run_id="colab_challenging_no_llm_demo",
    use_llm=False,
    use_adjudicator=False,
    audit_similarity_enabled=False,
)

print("run_id:", result.run_id)
print("run_dir:", result.run_dir)
print("documents:", len(result.documents))

## 6. Leer outputs JSON/JSONL

Los CSV son derivados. Para inspección y auditoría, lee JSON/JSONL o usa directamente `result.documents`.


In [ ]:
run_dir = Path("outputs/colab_challenging_no_llm_demo")

manifest = json.loads((run_dir / "run_manifest.json").read_text(encoding="utf-8"))
print("manifest run_id:", manifest["run_id"])
print("taxonomy:", manifest["taxonomy_version"])

print("\nUnidades y candidatos heurísticos:")
for line in (run_dir / "units.jsonl").read_text(encoding="utf-8").splitlines():
    unit = json.loads(line)
    candidates = [
        (item["label"], round(item["confidence"], 2)) for item in unit["heuristic_candidates"]
    ]
    print(unit["unit_id"], unit["final_notation"], candidates, unit["text"][:120])

print("\nAudit report preview:")
print((run_dir / "audit_report.md").read_text(encoding="utf-8")[:1000])

## 7. Utilidades de inspección del resultado

Estas funciones imprimen `ADN -> frase`, campos semánticos, evidencia, flags, relaciones, cadenas, auditoría y tiempos. Funcionan directamente con `result` o `llm_result`; no reconstruyen nada desde CSV.


In [ ]:
def value(x):
    """Convierte Enums/Pydantic values a texto legible."""
    return getattr(x, "value", x)


def values(xs):
    return [value(x) for x in xs]


def print_title(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def print_subtitle(title):
    print("\n" + "-" * 100)
    print(title)
    print("-" * 100)


def print_compact_adn(result):
    for doc in result.documents:
        print(f"\nDOCUMENTO: {doc.document_id}")
        print("=" * 80)
        for i, unit in enumerate(doc.units, start=1):
            print(f"ADN{i} {unit.final_notation} -> {unit.text}")


def print_evidence_spans(spans, fallback_text=None, indent="    "):
    if not spans:
        print(f"{indent}Evidencia: {fallback_text or '<sin evidence_spans explícitos>'}")
        return

    print(f"{indent}Evidencia:")
    for j, span in enumerate(spans, start=1):
        bits = []
        if span.char_start is not None and span.char_end is not None:
            bits.append(f"chars={span.char_start}-{span.char_end}")
        if span.start_ms is not None and span.end_ms is not None:
            bits.append(f"ms={span.start_ms}-{span.end_ms}")
        if span.source:
            bits.append(f"source={span.source}")
        meta = f" ({', '.join(bits)})" if bits else ""
        print(f"{indent}  {j}. {span.text}{meta}")


def print_unit(unit, idx):
    print(f"\nADN{idx} {unit.final_notation} -> {unit.text}")
    print(f"    unit_id: {unit.unit_id}")
    print(f"    índice: {unit.sequence_index}")
    print(f"    funciones: {values(unit.functions)}")
    print(f"    función primaria: {value(unit.primary_function)}")
    print(f"    funciones secundarias: {values(unit.secondary_functions)}")
    print(f"    certeza: {value(unit.certainty)}")
    print(f"    emoción expresada: {value(unit.emotion_expressed)}")
    print(f"    intensidad emocional: {unit.emotion_intensity}")
    print(f"    postura: {value(unit.stance)}")
    print(f"    confianza: {unit.confidence}")
    print(f"    método: {value(unit.method)}")
    print(f"    needs_review: {unit.needs_review}")
    print(f"    review_status: {value(unit.review_status)}")

    if unit.review_reasons:
        print("    razones de revisión:")
        for reason in unit.review_reasons:
            print(f"      - {reason}")

    print_evidence_spans(unit.evidence_spans, fallback_text=unit.text)

    if unit.heuristic_candidates:
        print("    candidatos heurísticos:")
        for candidate in unit.heuristic_candidates:
            print(
                f"      - {value(candidate.label)} "
                f"(confianza={candidate.confidence}): {candidate.reason}"
            )

    if unit.rejected_labels:
        print("    etiquetas rechazadas:")
        for rejected in unit.rejected_labels:
            print(f"      - {rejected.label} (confianza={rejected.confidence}): {rejected.reason}")

    if unit.validator_flags:
        print("    validator_flags:")
        for flag in unit.validator_flags:
            print(f"      - [{value(flag.severity)}] {flag.rule_id}: {flag.message}")


def print_document_summary(doc):
    print_title(f"DOCUMENTO: {doc.document_id}")
    print(f"source_path: {doc.source_path}")
    print(f"language: {doc.language}")
    print(f"unidades: {len(doc.units)}")
    print(f"relaciones: {len(doc.relations)}")
    print(f"cadenas: {len(doc.chains)}")
    print(f"segmentación: {doc.segmentation.strategy}")
    print(f"notas segmentación: {doc.segmentation.notes}")

    print_subtitle("ADN POR UNIDAD")
    for i, unit in enumerate(doc.units, start=1):
        print_unit(unit, i)

    unit_by_id = {unit.unit_id: unit for unit in doc.units}

    if doc.relations:
        print_subtitle("RELACIONES DETECTADAS")
        for rel in doc.relations:
            source = unit_by_id.get(rel.source_unit_id)
            target = unit_by_id.get(rel.target_unit_id)
            print(f"\n{rel.relation_id}")
            print(f"    tipo: {value(rel.relation_type)}")
            print(f"    confianza: {rel.confidence}")
            print(f"    método: {value(rel.method)}")
            print(f"    needs_review: {rel.needs_review}")
            print(f"    source: {rel.source_unit_id} -> {source.text if source else '<?> '}")
            print(f"    target: {rel.target_unit_id} -> {target.text if target else '<?> '}")
            print_evidence_spans(rel.evidence_spans)

    if doc.chains:
        print_subtitle("CADENAS NARRATIVAS")
        for chain in doc.chains:
            print(f"\n{chain.chain_id}")
            print(f"    tipo: {chain.chain_type}")
            print(f"    función narrativa: {chain.narrative_function}")
            print(f"    score: {chain.score}")
            print(f"    needs_review: {chain.needs_review}")
            print(f"    unidades: {chain.unit_ids}")
            print(f"    secuencia ADN: {' '.join(chain.notation_sequence)}")
            if chain.evidence_summary:
                print(f"    resumen evidencia: {chain.evidence_summary}")
            print("    frases de la cadena:")
            for unit_id in chain.unit_ids:
                unit = unit_by_id.get(unit_id)
                if unit:
                    print(f"      - {unit.final_notation} -> {unit.text}")


def print_output_files(result):
    print_title("ARCHIVOS GENERADOS")
    print(f"run_id: {result.run_id}")
    print(f"run_dir: {result.run_dir}")
    for name, output_path in result.output_paths.items():
        print(f"{name}: {output_path}")


def print_audit_report(result):
    audit_path = result.output_paths.get("audit_report")
    audit_path = audit_path or Path(result.run_dir) / "audit_report.json"
    if not Path(audit_path).exists():
        print_title("AUDITORÍA")
        print(f"No encontré audit_report.json en: {audit_path}")
        return

    report = json.loads(Path(audit_path).read_text(encoding="utf-8"))
    print_title("AUDITORÍA")
    print(f"run_id: {report.get('run_id')}")
    print(f"taxonomy_version_effective: {report.get('taxonomy_version_effective')}")
    print(f"prompt_version_effective: {report.get('prompt_version_effective')}")
    print(f"validator_version_effective: {report.get('validator_version_effective')}")

    print("\nsummary:")
    for key, val in report.get("summary", {}).items():
        print(f"  {key}: {val}")

    flags = report.get("validator_flags", [])
    print(f"\nvalidator_flags: {len(flags)}")
    for flag in flags[:20]:
        print(f"  - [{flag.get('severity')}] {flag.get('rule_id')}: {flag.get('message')}")

    conflicts = report.get("similarity_conflicts", [])
    print(f"\nsimilarity_conflicts: {len(conflicts)}")
    for conflict in conflicts[:20]:
        print(f"  - {conflict}")


def print_timing_report(result, top_n=20):
    timing_path = result.output_paths.get("timing_report")
    timing_path = timing_path or Path(result.run_dir) / "timing_report.json"
    if not Path(timing_path).exists():
        print_title("TIEMPOS")
        print(f"No encontré timing_report.json en: {timing_path}")
        return

    report = json.loads(Path(timing_path).read_text(encoding="utf-8"))
    print_title("TIEMPOS POR ETAPA")
    print(f"timing_report: {timing_path}")
    print(f"record_count: {report.get('record_count')}")

    summary = report.get("summary_by_stage", {})
    rows = sorted(summary.items(), key=lambda item: item[1].get("total_ms", 0), reverse=True)
    print("\nEtapas por tiempo total:")
    for stage, data in rows[:top_n]:
        print(
            f"  - {stage}: total={data.get('total_ms')} ms | "
            f"avg={data.get('avg_ms')} ms | max={data.get('max_ms')} ms | "
            f"count={data.get('count')}"
        )

    api_records = [r for r in report.get("records", []) if r.get("stage") == "openai.api_call"]
    print(f"\nLlamadas reales a OpenAI API: {len(api_records)}")
    for i, record in enumerate(api_records, start=1):
        meta = record.get("metadata", {})
        print(
            f"  {i}. {meta.get('profile_name')} | {meta.get('model')} | "
            f"attempt={meta.get('attempt')}/{meta.get('max_attempts')} | "
            f"duration={record.get('duration_ms')} ms"
        )
        print(f"     propósito: {meta.get('api_call_purpose')}")

    print("\nBottlenecks:")
    for record in report.get("bottlenecks", [])[:top_n]:
        meta = record.get("metadata", {})
        details = []
        for key in ("profile_name", "unit_id", "payload_chars", "cache_hit", "attempts"):
            if key in meta:
                details.append(f"{key}={meta[key]}")
        print(f"  - {record.get('stage')}: {record.get('duration_ms')} ms {' | '.join(details)}")


def print_pipeline_result(result, include_timing=True):
    print_title("RESULTADO DEL PIPELINE")
    print(f"run_id: {result.run_id}")
    print(f"run_dir: {result.run_dir}")
    print(f"documentos procesados: {len(result.documents)}")

    print_subtitle("ADN COMPACTO")
    print_compact_adn(result)

    for doc in result.documents:
        print_document_summary(doc)

    print_output_files(result)
    print_audit_report(result)
    if include_timing:
        print_timing_report(result)

## 8. Inspeccionar el resultado sin LLM


In [ ]:
print_pipeline_result(result, include_timing=False)

## 9. Ejecutar desde un archivo `.txt`


In [ ]:
Path("colab_challenging_example.txt").write_text(transcript_text, encoding="utf-8")
txt_result = run_pipeline(
    input_dir="colab_challenging_example.txt",
    output_dir="outputs",
    run_id="colab_txt_no_llm_demo",
    use_llm=False,
    use_adjudicator=False,
)
print(txt_result.run_dir)

## 10. Alternativa por CLI


In [ ]:
import subprocess

subprocess.run(
    [
        "narrative-dna",
        "run",
        "--input-dir",
        "colab_challenging_example.txt",
        "--output-dir",
        "outputs",
        "--run-id",
        "colab_cli_txt_no_llm",
        "--no-llm",
        "--no-adjudicator",
    ],
    check=True,
)
subprocess.run(["narrative-dna", "inspect", "--run-id", "colab_cli_txt_no_llm"], check=True)

## 11. Probar regresión golden


In [ ]:
!python -m pytest tests/test_golden_regression.py -q

## 12. Usar OpenAI desde Colab Secrets y ver tiempos por etapa

1. En Colab, abre **Secrets**.
2. Crea `OPENAI_API_KEY`.
3. Ejecuta esta celda.

Con `log_timings=True`, el pipeline imprime líneas `[timing]` durante la ejecución y además escribe `outputs/colab_challenging_llm_demo/timing_report.json`. Cada llamada real a OpenAI aparece como `stage=openai.api_call` e incluye `api_call_purpose` para explicar por qué se necesitó esa llamada.


In [ ]:
import os

from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

llm_result = run_pipeline_from_text(
    transcript_text,
    document_id="colab_challenging_demo_llm",
    source_path="<colab-string>",
    output_dir="outputs",
    run_id="colab_challenging_llm_demo",
    use_llm=True,
    use_adjudicator=True,
    audit_similarity_enabled=True,
    log_timings=True,
)

print("run_dir:", llm_result.run_dir)

## 13. Revisar significado, auditoría y tiempos del LLM

Esta celda usa directamente `llm_result.documents` y `llm_result.output_paths`. No usa CSV como fuente de verdad.


In [ ]:
print_pipeline_result(llm_result, include_timing=True)

## 14. Comparar con el texto corto de caja negra emocional

Ejecuta esto si quieres contrastar el caso desafiante contra el texto corto original. Si el modelo devuelve demasiados `N`, el problema ya no es el texto: hay que revisar prompt, taxonomía, segmentación o adjudicación.


In [ ]:
# black_box_result = run_pipeline_from_text(
#     black_box_memory_text,
#     document_id="colab_black_box_memory_llm",
#     source_path="<colab-string>",
#     output_dir="outputs",
#     run_id="colab_black_box_memory_llm_demo",
#     use_llm=True,
#     use_adjudicator=True,
#     audit_similarity_enabled=True,
#     log_timings=True,
# )
# print_pipeline_result(black_box_result, include_timing=True)


## 15. Evaluar contra synthetic gold high-confidence

Sólo `synthetic_gold_high_confidence` debe usarse como regresión.


In [ ]:
# gold = "outputs/colab_challenging_llm_demo/synthetic_gold_high_confidence.jsonl"
# !narrative-dna evaluate --run-id colab_challenging_llm_demo --gold {gold}
